In [1]:
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
import random
import shutil
import os

def load_data():
    data=load_dataset("hotpotqa/hotpot_qa","distractor")
    """
    loads the hotpot-qa dataset that comes with prebuilt questions, paragrpahs, answers used for testing how well RAG 
    system can find and use relevant information. there are other datsets as well like NQ etc. we are using distractor 
    version of it where for each question it provides 10 paragraphs with some relevant info and some irrelvant info and tests
     the system retireval. the fullwiki version means we need to look at wikipedia data dump without providing the context 
     which is harder.  
    """
    return data

def extract_paragraphs(data,num_rows):
    paragraphs=[]
    for i in range(num_rows):
        titles=data["train"][i]["context"]["title"]
        sentences=data["train"][i]["context"]["sentences"]
        for title, sent_list in zip(titles,sentences):
            joined_sentence="".join(sent_list)
            full_text=title+" : "+joined_sentence
            paragraphs.append(full_text)
    return paragraphs

def chunk_paragraphs(paragraphs):
    Text_Splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
    """ 
    here chunk size refers to number of characters that will be sotred as one chunk.
    if the character limit is reached mid-word it extends till the end of word since its recruseve text splitter.
    chunk overlap refers to number of characters taken from previous chunk for the context. 
    """
    chunks=[]
    for paragraph in paragraphs:
        temp_chunk=Text_Splitter.split_text(paragraph)
        chunks.extend(temp_chunk)
    return chunks

def get_embedding_model():
    Embedding_Model_Name="sentence-transformers/all-mpnet-base-v2"
    Embedding_Model=HuggingFaceEmbeddings(model_name=Embedding_Model_Name)
    return Embedding_Model

def store_chunks(chunks,Embedding_Model):
    #clearing old db first so we dont get duplicate/mismatched ids issue
    if os.path.exists("./Chroma_DB"):
        shutil.rmtree("./Chroma_DB")
    Unique_IDs=[str(i) for i in range(len(chunks))]
    Vector_Storage=Chroma.from_texts(texts=chunks, embedding=Embedding_Model,persist_directory="./Chroma_DB",ids=Unique_IDs)
    return Vector_Storage

def get_random_query(data,num_rows):
    random_row=random.randint(0,num_rows-1)
    query=data["train"][random_row]["question"]
    correct_answer=data["train"][random_row]["answer"]
    print("Row:",random_row)
    print("Question:",query)
    print("Correct Answer:",correct_answer)
    return query,correct_answer

def retrieve(Vector_Storage,query,k=5):
    results=Vector_Storage.similarity_search_with_relevance_scores(query,k=k) #here k means gives the top these number of chunks
    for doc,score in results:
        print(doc.page_content)
        print("Relevance Score:",score,"\n<------------------------------>")
    return results

def generate_answer(results,query):
    context_pieces = []
    for doc, score in results:
        context_pieces.append(doc.page_content)
    context_text = "\n\n".join(context_pieces)
    prompt = f"""Use the context below to answer the question. The context may contain sentence fragments that continue across lines.
Context:
{context_text}
Question: {query}
Answer:"""
    llm = ChatOllama(model="qwen3.5:9b", num_predict=8000) #selecting the model as well as the number of output token size
    response = llm.invoke(prompt)
    return response.content


# running the whole pipeline
num_rows=50

data=load_data()
paragraphs=extract_paragraphs(data,num_rows)
chunks=chunk_paragraphs(paragraphs)
Embedding_Model=get_embedding_model()
Vector_Storage=store_chunks(chunks,Embedding_Model)

query,correct_answer=get_random_query(data,num_rows)
results=retrieve(Vector_Storage,query)
answer=generate_answer(results,query)

print(answer)
print("Actual Answer: "+correct_answer)

c:\Users\muhammad kashan\Desktop\RAG PipeLine ICT Project B\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10074.43it/s]


Row: 10
Question: Fast Cars, Danger, Fire and Knives includes guest appearances from which hip hop record executive?
Correct Answer: Jaime Meline
in addition to guest spots from Del the Funky Homosapien, King T, B-Real from Cypress Hill, Knoc-turn'al, Khujo from Goodie Mob, among others.
Relevance Score: 0.4560161053601972 
<------------------------------>
Fast Cars, Danger, Fire and Knives : Fast Cars, Danger, Fire and Knives is an EP by American hip hop artist Aesop Rock. Released via the Definitive Jux label on February 22, 2005, the record is
Relevance Score: 0.44422594595775966 
<------------------------------>
Entirely produced by Beatnick & K-Salaam, the 6-song EP is a blend between reggae and hip hop, and includes guest appearances from Junior Reid, Jr. Kelly, Jahdan Blakkamoore, and DJ Full Factor. The
Relevance Score: 0.3880814968536701 
<------------------------------>
The album was released on January 21, 2014 by Rhymesayers Entertainment. The record was produced entirely b